# Preprocessed Mel + JSONL로 Whisper Base LoRA 학습

입력:

```text
/home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed/
├── features/*.npy
└── manifest.jsonl
```

모델과 학습 방식:

- Base model: `openai/whisper-base`
- Language: Korean
- Task: transcription
- LoRA target: encoder/decoder Attention의 `q_proj`, `v_proj`
- 기본 LoRA 설정: rank 16, alpha 32, dropout 0.05
- 입력 특징: `(80, 3000)` float32 Log-Mel

이 노트북은 NPY에 feature extractor를 다시 적용하지 않습니다. 저장된 Log-Mel을 Whisper Encoder에 바로 전달합니다.


## 0. 패키지 설치

필요한 경우 주석을 해제해 실행한 뒤 커널을 재시작하세요.


In [1]:
%pip install -U peft accelerate safetensors jiwer


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

import peft
import transformers
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


/home/lmh/project/whisper/1stlayer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.1+cu130
transformers: 5.13.1
peft: 0.19.1
CUDA available: True
GPU: NVIDIA GeForce RTX 5080


## 1. 경로와 학습 설정


In [3]:
PREPROCESSED_ROOT = Path(
    '/home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed'
)
FEATURE_DIR = PREPROCESSED_ROOT / 'features'
MANIFEST_PATH = PREPROCESSED_ROOT / 'manifest.jsonl'

MODEL_ID = 'openai/whisper-small'
LANGUAGE = 'korean'
TASK = 'transcribe'

OUTPUT_DIR = Path(
    '/home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_small' \
    '_lora_output'
)

# manifest의 label_token_ids보다 text를 현재 Whisper-base tokenizer로 다시 변환하는 것이 안전합니다.
USE_PRETOKENIZED_LABELS = False

# manifest에 split 정보가 없을 때 검증 데이터 비율
EVAL_RATIO = 0.10
RANDOM_SEED = 42

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'v_proj']

# Training
NUM_TRAIN_EPOCHS = 30
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
GRADIENT_CHECKPOINTING = True
DATALOADER_NUM_WORKERS = 2

# None이면 처음부터 학습합니다. 예: OUTPUT_DIR / 'checkpoint-100'
RESUME_FROM_CHECKPOINT: Optional[Path] = None

# 학습 후 선택 사항
RUN_WER_EVALUATION = True
MAX_WER_SAMPLES = 100
MERGE_ADAPTER_AFTER_TRAINING = False


## 2. Manifest 읽기와 NPY 경로 해결

지원하는 manifest 주요 필드:

```json
{
  "id": "000_G0102_chunk_00000",
  "feature_path": "features/000_G0102_chunk_00000.npy",
  "text": "전사 문장"
}
```

`feature_path`가 절대경로, manifest 기준 상대경로, 파일명만 저장된 경우를 모두 확인합니다.


In [4]:
def read_jsonl(path: Path) -> List[dict]:
    if not path.is_file():
        raise FileNotFoundError(f'manifest가 없습니다: {path}')

    records: List[dict] = []
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'{path}:{line_number}: 잘못된 JSON') from exc

            if not record.get('text', '').strip():
                raise ValueError(f'{path}:{line_number}: text가 비어 있습니다.')
            records.append(record)

    if not records:
        raise ValueError(f'manifest 데이터가 없습니다: {path}')
    return records


def resolve_feature_path(record: dict) -> Path:
    raw_path = record.get('feature_path')

    candidates: List[Path] = []
    if raw_path:
        raw_path = Path(str(raw_path))
        if raw_path.is_absolute():
            candidates.append(raw_path)
        else:
            candidates.extend(
                [
                    PREPROCESSED_ROOT / raw_path,
                    FEATURE_DIR / raw_path,
                    FEATURE_DIR / raw_path.name,
                ]
            )

    record_id = record.get('id') or record.get('file_id')
    if record_id:
        candidates.append(FEATURE_DIR / f'{record_id}.npy')

    # 순서를 보존하며 중복 후보 제거
    unique_candidates = list(dict.fromkeys(path.resolve() for path in candidates))
    for candidate in unique_candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        'NPY 특징 파일을 찾을 수 없습니다.\n'
        f"record id={record.get('id', record.get('file_id'))}\n"
        + '\n'.join(str(path) for path in unique_candidates)
    )


records = read_jsonl(MANIFEST_PATH)
print('manifest records:', len(records))
print('first record keys:', sorted(records[0].keys()))
print('first feature:', resolve_feature_path(records[0]))
print('first text:', records[0]['text'])


manifest records: 31
first record keys: ['audio_path', 'crop_duration', 'crop_end', 'crop_start', 'feature_path', 'genders', 'id', 'label_token_ids', 'n_frames', 'n_mels', 'segments', 'source_label', 'source_wav', 'speakers', 'text']
first feature: /home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed/features/000_A0051_S0001_0_G0101_chunk_00000.npy
first text: 매직 데이터 그 요즘 방 탈출 카페를 좋아하는데 요즘 그래서 그 창업 쪽으로 한번 생각을 해 보고 있어요. 그 방 탈출 카페라고 해서 이제 창업 쪽도 그냥 생각을 하는데 그냥 카페를 창업하고 싶어 하는 사람들도 많고 그래서 요즘은 좀 이런 생각들도 많이 하고 있어.


## 3. Train / Eval 분리

우선 `split`, `dataset_type`, `requested_split` 필드를 확인합니다. 명시적인 train/eval 구성이 없으면 `source_wav` 단위 분리를 시도합니다. 원본 WAV가 하나뿐이면 마지막 수단으로 chunk 단위 무작위 분리를 사용합니다.


In [5]:
def get_explicit_split(record: dict) -> Optional[str]:
    for key in ('split', 'dataset_type', 'requested_split'):
        value = record.get(key)
        if value is not None:
            value = str(value).strip().lower()
            if value in {'train', 'development', 'validation', 'eval', 'test'}:
                return value
    return None


def random_record_split(records: Sequence[dict]) -> Tuple[List[dict], List[dict]]:
    if len(records) < 2:
        raise ValueError('Train/Eval 분리를 위해 레코드가 2개 이상 필요합니다.')
    shuffled = list(records)
    random.Random(RANDOM_SEED).shuffle(shuffled)
    eval_size = max(1, round(len(shuffled) * EVAL_RATIO))
    if eval_size >= len(shuffled):
        eval_size = 1
    return shuffled[eval_size:], shuffled[:eval_size]


def split_train_eval(records: Sequence[dict]) -> Tuple[List[dict], List[dict], str]:
    explicit_train = [record for record in records if get_explicit_split(record) == 'train']
    explicit_eval = [
        record
        for record in records
        if get_explicit_split(record) in {'development', 'validation', 'eval', 'test'}
    ]
    if explicit_train and explicit_eval:
        return explicit_train, explicit_eval, 'manifest explicit split'

    groups: Dict[str, List[dict]] = {}
    for record in records:
        group_key = str(
            record.get('source_wav')
            or record.get('source_audio')
            or record.get('speaker')
            or ''
        )
        if group_key:
            groups.setdefault(group_key, []).append(record)

    if len(groups) >= 2 and sum(len(group) for group in groups.values()) == len(records):
        keys = list(groups)
        random.Random(RANDOM_SEED).shuffle(keys)
        target_eval_count = max(1, round(len(records) * EVAL_RATIO))
        eval_keys: List[str] = []
        eval_count = 0
        for key in keys:
            if eval_count >= target_eval_count and eval_keys:
                break
            eval_keys.append(key)
            eval_count += len(groups[key])

        eval_key_set = set(eval_keys)
        train = [item for key in keys if key not in eval_key_set for item in groups[key]]
        eval_items = [item for key in keys if key in eval_key_set for item in groups[key]]
        if train and eval_items:
            return train, eval_items, 'source-level split'

    train, eval_items = random_record_split(records)
    return train, eval_items, 'random chunk split (leakage 주의)'


train_records, eval_records, split_method = split_train_eval(records)
print('split method:', split_method)
print('train:', len(train_records))
print('eval:', len(eval_records))


split method: random chunk split (leakage 주의)
train: 28
eval: 3


## 4. Whisper Processor와 Base model 불러오기


In [6]:
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language=LANGUAGE,
    task=TASK,
)

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task = TASK
base_model.generation_config.forced_decoder_ids = None
base_model.config.use_cache = False

EXPECTED_N_MELS = int(base_model.config.num_mel_bins)
EXPECTED_N_FRAMES = int(base_model.config.max_source_positions * 2)
MAX_TARGET_POSITIONS = int(base_model.config.max_target_positions)

print('n_mels:', EXPECTED_N_MELS)
print('n_frames:', EXPECTED_N_FRAMES)
print('max target positions:', MAX_TARGET_POSITIONS)
print('decoder start token:', base_model.config.decoder_start_token_id)


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1397.13it/s]


n_mels: 80
n_frames: 3000
max target positions: 448
decoder start token: 50258


## 5. NPY 특징과 text label Dataset


In [7]:
class MelJsonDataset(Dataset):
    def __init__(self, records: Sequence[dict], processor: WhisperProcessor):
        self.records = list(records)
        self.tokenizer = processor.tokenizer

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        record = self.records[index]
        feature_path = resolve_feature_path(record)
        mel = np.load(feature_path, allow_pickle=False).astype(np.float32, copy=False)

        expected_shape = (EXPECTED_N_MELS, EXPECTED_N_FRAMES)
        if mel.shape != expected_shape:
            raise ValueError(
                f'{feature_path}: shape={mel.shape}, expected={expected_shape}'
            )
        if not np.isfinite(mel).all():
            raise ValueError(f'{feature_path}: NaN 또는 무한대가 있습니다.')

        labels = None
        if USE_PRETOKENIZED_LABELS:
            stored = record.get('label_token_ids') or record.get('labels')
            if isinstance(stored, list) and stored:
                labels = [int(token) for token in stored]

        if labels is None:
            labels = self.tokenizer(
                record['text'].strip(), add_special_tokens=True
            ).input_ids

        if len(labels) > MAX_TARGET_POSITIONS:
            raise ValueError(
                f"{record.get('id', index)}: labels={len(labels)} tokens, "
                f'max={MAX_TARGET_POSITIONS}. chunk를 더 작게 나누세요.'
            )

        return {
            'input_features': mel,
            'labels': labels,
            'id': str(record.get('id', record.get('file_id', index))),
            'text': record['text'].strip(),
        }


train_dataset = MelJsonDataset(train_records, processor)
eval_dataset = MelJsonDataset(eval_records, processor)

sample = train_dataset[0]
print('sample id:', sample['id'])
print('input shape:', sample['input_features'].shape)
print('input dtype:', sample['input_features'].dtype)
print('labels length:', len(sample['labels']))
print('decoded:', processor.tokenizer.decode(sample['labels'], skip_special_tokens=True))


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


sample id: 000_A0051_S0001_0_G0101_chunk_00024
input shape: (80, 3000)
input dtype: float32
labels length: 71
decoded: 큰 가게를 목적으로 하는 사람들도 있겠지만은 작은 가게를 목적으로 하고 나가는 사람들도 있- 있지 않을까? 그래서 그 큰 가게도 결국에는 매출이 떨어지고 그렇게 함으로써 그 상권은 망할 거라고 생각을 해. 나는 [SONANT] 응 음


## 6. LoRA 적용

`q_proj`, `v_proj`는 encoder self-attention, decoder self-attention, decoder cross-attention 전체에서 이름이 같은 projection에 적용됩니다. 원본 Whisper 가중치는 고정되고 `lora_A`, `lora_B`만 업데이트됩니다.

수식:

$$W = W_0 + \frac{\alpha}{r}BA$$


In [8]:
matched_targets = [
    name
    for name, module in base_model.named_modules()
    if name.endswith(tuple(LORA_TARGET_MODULES))
]
if not matched_targets:
    raise ValueError(f'LoRA target을 찾지 못했습니다: {LORA_TARGET_MODULES}')

print('matched target modules:', len(matched_targets))
for name in matched_targets[:20]:
    print(' ', name)

lora_config = LoraConfig(
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable_names = [
    name for name, parameter in model.named_parameters() if parameter.requires_grad
]
if not trainable_names:
    raise RuntimeError('학습 가능한 LoRA 파라미터가 없습니다.')
if not all('lora_' in name for name in trainable_names):
    print('주의: LoRA 이외의 trainable parameter가 있습니다.')

print('trainable tensor count:', len(trainable_names))
for name in trainable_names[:20]:
    print(' ', name)


matched target modules: 72
  model.encoder.layers.0.self_attn.v_proj
  model.encoder.layers.0.self_attn.q_proj
  model.encoder.layers.1.self_attn.v_proj
  model.encoder.layers.1.self_attn.q_proj
  model.encoder.layers.2.self_attn.v_proj
  model.encoder.layers.2.self_attn.q_proj
  model.encoder.layers.3.self_attn.v_proj
  model.encoder.layers.3.self_attn.q_proj
  model.encoder.layers.4.self_attn.v_proj
  model.encoder.layers.4.self_attn.q_proj
  model.encoder.layers.5.self_attn.v_proj
  model.encoder.layers.5.self_attn.q_proj
  model.encoder.layers.6.self_attn.v_proj
  model.encoder.layers.6.self_attn.q_proj
  model.encoder.layers.7.self_attn.v_proj
  model.encoder.layers.7.self_attn.q_proj
  model.encoder.layers.8.self_attn.v_proj
  model.encoder.layers.8.self_attn.q_proj
  model.encoder.layers.9.self_attn.v_proj
  model.encoder.layers.9.self_attn.q_proj
trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267
trainable tensor count: 144
  base_model.model.model.enc

## 7. Speech Seq2Seq Data Collator

Log-Mel은 동일한 shape으로 stack하고, text label은 batch 최대 길이로 padding한 뒤 PAD를 `-100`으로 바꿔 loss에서 제외합니다.


In [9]:
@dataclass
class DataCollatorWhisperLoRA:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = torch.from_numpy(
            np.stack([feature['input_features'] for feature in features])
        ).float()

        label_features = [
            {'input_ids': feature['labels']} for feature in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch['attention_mask'].ne(1), -100
        )

        # Whisper model이 decoder start token을 자동으로 넣으므로 중복 시작 token을 제거합니다.
        if (
            labels.shape[1] > 0
            and (labels[:, 0] == self.decoder_start_token_id).all().item()
        ):
            labels = labels[:, 1:]

        return {'input_features': input_features, 'labels': labels}


data_collator = DataCollatorWhisperLoRA(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

test_features = [train_dataset[i] for i in range(min(2, len(train_dataset)))]
test_batch = data_collator(test_features)
print('batch input:', test_batch['input_features'].shape)
print('batch labels:', test_batch['labels'].shape)
print('ignored label positions:', int((test_batch['labels'] == -100).sum()))


batch input: torch.Size([2, 80, 3000])
batch labels: torch.Size([2, 106])
ignored label positions: 36


## 8. 학습 전 Forward/Backward smoke test

전체 학습 전에 batch 하나로 loss가 유한한지, LoRA 파라미터에 gradient가 생성되는지 검사합니다.


In [10]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(DEVICE)
model.train()
model.zero_grad(set_to_none=True)

smoke_batch = {key: value.to(DEVICE) for key, value in test_batch.items()}
smoke_output = model(**smoke_batch)
print('smoke loss:', float(smoke_output.loss.detach().cpu()))
print('logits:', tuple(smoke_output.logits.shape))
assert torch.isfinite(smoke_output.loss), 'loss가 NaN 또는 Inf입니다.'

smoke_output.loss.backward()
gradient_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and parameter.grad is not None
]
print('LoRA tensors with gradient:', len(gradient_names))
if not gradient_names:
    raise RuntimeError('LoRA gradient가 생성되지 않았습니다.')

model.zero_grad(set_to_none=True)
del smoke_output, smoke_batch
if torch.cuda.is_available():
    torch.cuda.empty_cache()


smoke loss: 0.7098848223686218
logits: (2, 106, 51865)
LoRA tensors with gradient: 144


## 9. Trainer 설정

기본 effective batch size는 GPU 1장 기준 `2 × 8 = 16`입니다. Base 모델과 GPU 메모리에 따라 `PER_DEVICE_TRAIN_BATCH_SIZE`를 조정하세요.


In [11]:
HAS_CUDA = torch.cuda.is_available()
USE_BF16 = HAS_CUDA and getattr(
    torch.cuda, 'is_bf16_supported', lambda: False
)()
USE_FP16 = HAS_CUDA and not USE_BF16

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='linear',
    max_grad_norm=1.0,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    fp16=USE_FP16,
    bf16=USE_BF16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=10,
    save_total_limit=2,
    predict_with_generate=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    remove_unused_columns=False,
    label_names=['labels'],
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=HAS_CUDA,
    optim='adamw_torch',
    report_to='none',
    push_to_hub=False,
    seed=RANDOM_SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor,
)

print('fp16:', USE_FP16, 'bf16:', USE_BF16)
print('effective batch size per GPU:', (
    PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
))


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


fp16: False bf16: True
effective batch size per GPU: 16


## 10. LoRA 학습 실행

각 optimizer step에서 원본 Whisper 가중치는 고정되고 LoRA `A`, `B`만 업데이트됩니다. Evaluation은 우선 생성 없이 validation loss를 사용하므로 학습 중 메모리와 시간이 절약됩니다.


In [12]:
resume_path = None
if RESUME_FROM_CHECKPOINT is not None:
    resume_path = str(Path(RESUME_FROM_CHECKPOINT))

train_result = trainer.train(resume_from_checkpoint=resume_path)
print(train_result)


Epoch,Training Loss,Validation Loss
1,No log,0.806274
2,No log,0.785880
3,No log,0.763684
4,No log,0.739579
5,5.104864,0.717574
6,5.104864,0.695004
7,5.104864,0.677321
8,5.104864,0.659622
9,5.104864,0.644372
10,4.208459,0.630286


TrainOutput(global_step=60, training_loss=3.5234247207641602, metrics={'train_runtime': 60.6608, 'train_samples_per_second': 13.847, 'train_steps_per_second': 0.989, 'total_flos': 2.445520896e+17, 'train_loss': 3.5234247207641602, 'epoch': 30.0})


## 11. 최종 Adapter와 Processor 저장

저장되는 것은 Whisper 전체 모델이 아니라 작은 LoRA adapter입니다. 추론할 때 동일한 `openai/whisper-base`가 필요합니다.


In [13]:
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
processor.save_pretrained(FINAL_ADAPTER_DIR)

training_summary = {
    'base_model': MODEL_ID,
    'train_records': len(train_dataset),
    'eval_records': len(eval_dataset),
    'split_method': split_method,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'lora_targets': LORA_TARGET_MODULES,
}
with (FINAL_ADAPTER_DIR / 'training_summary.json').open('w', encoding='utf-8') as file:
    json.dump(training_summary, file, ensure_ascii=False, indent=2)

print('adapter saved:', FINAL_ADAPTER_DIR.resolve())
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


adapter saved: /home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_small_lora_output/final_adapter
{
  "base_model": "openai/whisper-small",
  "train_records": 28,
  "eval_records": 3,
  "split_method": "random chunk split (leakage 주의)",
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_targets": [
    "q_proj",
    "v_proj"
  ]
}


## 12. 학습된 Adapter로 샘플 추론


In [14]:
model.eval()
model.config.use_cache = True
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

inference_sample = eval_dataset[0]
inference_features = torch.from_numpy(
    inference_sample['input_features']
).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features=inference_features,
        max_new_tokens=225,
    )

prediction = processor.batch_decode(
    predicted_ids, skip_special_tokens=True
)[0]
print('id:', inference_sample['id'])
print('reference:', inference_sample['text'])
print('prediction:', prediction)


[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cus

id: 000_A0051_S0001_0_G0101_chunk_00019
reference: 뭔가 더 자기네들은 매출은 오르는 느낌인 거고 그치. 그치. 이게 그- 어- 어- 경제적으로 보면 어- 어떻게 보면은 한 쪽은 내려가지만 어- 어느 쪽은 올라갈 수 밖에 없는 경우인 거잖아. 경제라는 게 어떻게 비- 그런 상황이고 뭐 대체재라는 게 있- 있고 그러니까 그래서 일단은
prediction:  뭔가 더 자기네들은 매출 노르는 느낌인 거고 그치 그치 이게 그거 어떻게 경제적으로 보면 어떻게 보면 한쪽은 내려가지 마 어느 쪽은 올라갈 수밖에 없는 경우인 거잖아? 경제라는 게 어떻게 비 그런 상황이 뭐 대체제 라는 게 있고 그러니까 그래서 일단은


In [15]:
list(eval_dataset)

[{'input_features': array([[-0.6972326 , -0.18368769, -0.21304369, ..., -0.6972326 ,
          -0.6972326 , -0.6972326 ],
         [-0.5489055 , -0.25190628, -0.22864485, ..., -0.6972326 ,
          -0.6972326 , -0.6972326 ],
         [-0.6972326 , -0.3306259 , -0.41774416, ..., -0.6972326 ,
          -0.6972326 , -0.6972326 ],
         ...,
         [-0.6972326 , -0.6972326 , -0.6972326 , ..., -0.6972326 ,
          -0.6972326 , -0.6972326 ],
         [-0.6972326 , -0.6972326 , -0.6972326 , ..., -0.6972326 ,
          -0.6972326 , -0.6972326 ],
         [-0.6972326 , -0.6972326 , -0.6972326 , ..., -0.6972326 ,
          -0.6972326 , -0.6972326 ]], shape=(80, 3000), dtype=float32),
  'labels': [50258,
   50264,
   50359,
   50363,
   167,
   255,
   13833,
   6990,
   5650,
   7091,
   226,
   2990,
   22571,
   17591,
   24591,
   2124,
   10258,
   39402,
   12652,
   4215,
   3675,
   1313,
   4296,
   8464,
   13,
   4296,
   8464,
   13,
   10496,
   4296,
   12,
   9076,
   12,
 

## 13. 선택 사항: Validation WER

생성 평가는 시간이 오래 걸릴 수 있어 기본값은 꺼져 있습니다. `RUN_WER_EVALUATION=True`로 변경하면 최대 `MAX_WER_SAMPLES`개를 평가합니다.


In [16]:
if RUN_WER_EVALUATION:
    from jiwer import wer

    predictions: List[str] = []
    references: List[str] = []
    eval_count = min(MAX_WER_SAMPLES, len(eval_dataset))

    model.eval()
    for index in range(eval_count):
        sample = eval_dataset[index]
        features = torch.from_numpy(sample['input_features']).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output_ids = model.generate(
                input_features=features, max_new_tokens=225
            )
        text = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
        predictions.append(text)
        references.append(sample['text'])

    validation_wer = 100.0 * wer(references, predictions)
    print(f'WER ({eval_count} samples): {validation_wer:.2f}%')
else:
    print('WER evaluation skipped')


[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WER (3 samples): 27.50%


## 14. 선택 사항: LoRA를 Base model에 merge

Merge하면 adapter를 별도로 로드하지 않아도 되지만 저장 크기는 전체 Whisper Base 크기가 됩니다. GPU/CPU 메모리를 추가로 사용할 수 있습니다.


In [17]:
if MERGE_ADAPTER_AFTER_TRAINING:
    MERGED_MODEL_DIR = OUTPUT_DIR / 'merged_model'
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
    processor.save_pretrained(MERGED_MODEL_DIR)
    print('merged model saved:', MERGED_MODEL_DIR.resolve())
else:
    print('adapter merge skipped')


adapter merge skipped


## 저장한 Adapter를 새 프로세스에서 불러오는 코드

```python
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

base = WhisperForConditionalGeneration.from_pretrained('openai/whisper-base')
model = PeftModel.from_pretrained(base, FINAL_ADAPTER_DIR)
processor = WhisperProcessor.from_pretrained(FINAL_ADAPTER_DIR)
model.eval().to('cuda')
```
